# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Load the metadata object
metadata = dataset.metadata

print("Dataset Name: ", metadata.name)
print("Dataset Description: ", metadata.description)
print("Dataset DOI Identifier: ", getattr(metadata, 'identifier', 'N/A'))
print("Dataset Version: ", getattr(metadata, 'version', 'N/A'))

## 2. Data Overview
Let's review available record sets in the dataset. For each record set, we will print its `@id`, `name`, and field `@id`s (columns).

**Note:** All references to entities are made using their `@id` as required.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    # List all columns/fields in this record set
    print("  Fields and their @id's:")
    for field in rs.fields:
        print(f"    - {field.id}: {field.name}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We will select the main record set (identified above) and extract all its records as a DataFrame.

All entity names are referenced by their `@id`.

In [ ]:
# Extract data from all record sets found
dataframes = {}

# We'll use the main record set for proteomics data (first one for demonstration)
record_set_ids = [rs.id for rs in record_sets]

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"RecordSet '{rs.id}' loaded: {df.shape[0]} records, {df.shape[1]} columns.")

# Display columns for the first record set
main_record_set_id = record_set_ids[0]
print(f"Available columns in main record set (@id: {main_record_set_id}):\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we select a numeric field from the main record set (by its `@id`), filter on that field, and perform basic normalization and grouping.

In [ ]:
# (Adjust these @ids based on your printed fields in Section 2 / previous cell)
df = dataframes[main_record_set_id]

# As an example, let's assume the main quantitative field is 'coverage_percent' (replace with actual @id if different).
# Find a suitable numeric field for demonstration:
numeric_field_candidates = [col for col in df.columns if df[col].dtype in [int, float] or pd.api.types.is_numeric_dtype(df[col])]
print("Numeric columns found:", numeric_field_candidates)
if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # Use first numeric field found
else:
    # If no numeric columns detected by dtype, pick one likely numeric (try 'coverage_percent', 'peptide_count', 'MW' etc.)
    possible = ['coverage_percent', 'peptide_count', 'MW', 'abundance']
    numeric_field = [col for col in possible if col in df.columns]
    if numeric_field:
        numeric_field = numeric_field[0]
    else:
        numeric_field = df.columns[0]  # fallback to first column

print(f"Using field '@id': {numeric_field} for numeric analysis.")

# Choose a threshold for filtering
threshold = df[numeric_field].quantile(0.9) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10

filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df.head(5)
print(f"Filtered records where {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
    print(f"Normalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

# Pick a group field by @id (as an example, try 'description', 'accession', etc.)
group_field_candidates = [f for f in df.columns if f != numeric_field]
group_field = group_field_candidates[0] if group_field_candidates else numeric_field
print(f"Attempting to group by: {group_field}")

if group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by '{group_field}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. For example, let's plot the distribution of the selected numeric field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the selected numeric field
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# If normalization column was computed
if numeric_field + '_normalized' in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field + '_normalized'].dropna(), kde=True)
    plt.title(f"Normalized Distribution of {numeric_field} (filtered records)")
    plt.xlabel(numeric_field + '_normalized')
    plt.show()

# Scatter plot: numeric field vs. another field, if available
if len(df.columns) > 1:
    y_field = [col for col in df.columns if col != numeric_field and pd.api.types.is_numeric_dtype(df[col])]
    if y_field:
        y_field = y_field[0]
        plt.figure(figsize=(6,6))
        sns.scatterplot(data=df, x=numeric_field, y=y_field)
        plt.title(f"{numeric_field} vs {y_field}")
        plt.xlabel(numeric_field)
        plt.ylabel(y_field)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

- We loaded dataset metadata and records by referencing all entities via their `@id` as required by Croissant.
- We listed all available record sets and their respective fields (columns).
- We performed simple exploratory analysis: filtering, normalization, grouping, and visualization of data fields.

You can further extend this workflow by processing other record sets, exploring relationships between columns, and building machine learning models on the processed data as needed.